In [ ]:
%load_ext autoreload
%autoreload 2
# %cd /pscratch/sd/b/brenthu/chem_llm
%cd /nfs/roberts/project/pi_vsb4/byh2/chem_llm

import json
import os
from chem_llm import config
from dotenv import load_dotenv
load_dotenv()
    
# HF_HOME must be set before transformers is imported so it picks up the cache dir.
os.environ["HF_HOME"] = config.HF_HOME
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from chem_llm.agent_core import run_agent

In [ ]:
os.makedirs(config.WORK_DIR, exist_ok=True)
os.chdir(config.WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [4]:
# LOAD MODEL

if not config.HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is not set. Run `export HF_TOKEN=hf_xxx` before launching main.py."
    )
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, token=config.HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
    token=config.HF_TOKEN,
)

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

In [12]:
# TASK = """
# Write two new Python scripts based off of the pre-existing generate_structures.py and setup_jobs.py that generates perturbed TiO2 crystal structures using pymatgen and input files for quantum espresso.

# Required workflow:
# 1. Read the existing example/generate_structures.py and note down how each constant and function works as well as anything you don't need for TiO2 (for example, interpolating between phase transitions)
# 2. Use the generate_cif tool to obtain the base rutile TiO4 crystal structure.
# 3. In the python script, use pymatgen to generate 50 additional structures for 51 total structures.
# 4. The perturbations should be chemically reasonable random displacements of the Ti and O atoms at room temperature.
# 5. Then read the existing example/setup_jobs.py, noting down what it does and any changes you must make to work with the new workflow
# 6. Use the get_pseudopotential tool to create your own template folder. Please use norm conserving, fully relativisitc, pbesol, stringent pseudopotentials
# 6. Then write a new python file. Leave SLURM account, email, etc. blank

# Requirements:
# - Base the perturbations on the endpoint structure obtained from generate_cif.
# - Organize the output files in a clear directory structure.
# - Comment the code where nontrivial crystallographic operations are performed.

# Before calling done:
# - Execute generate_structures.py and setup_jobs.py.
# - Verify it completes without errors.
# - Read at least one intermediate CIF and one quantum espresso input file to confirm it appears reasonable.

# In the done summary, include:
# - The Materials Project IDs used for the endpoint structures.
# - An outline of how the new python workflow works.
# - Any scientific assumptions made and a quick justification.
# - Any details that might require further expert domain knowledge.
# """

In [26]:
TASK = """
You are a quantum chemist running DFT calculations for CsPbBr3.
Do NOT try to generate a cif file.
Use the get_pseudopotential tool to get material appropriate pseudopotentials.
Determine whether, for the material that is being considered, there are elements requiring fully relativistic pseudopotentials.
Otherwise, use scalar relativstic pseudopotentials. The same pseudopotential must be used consistently for all atoms in the material.
Finally, write a pw.x input file for quantum espresso.
"""

In [27]:
final_state = run_agent(TASK, model, tokenizer, verbose=True)

print("\nFINAL STATE:\n", json.dumps(final_state.to_dict(), indent=2))


=== STEP 1 ===
RAW MODEL OUTPUT:
 { "tool": "scan_pymatgen_docs", "parameters": { "query": "Structure.get_space_group_info" } }
PARSED TOOL CALL:
 {'tool': 'scan_pymatgen_docs', 'parameters': {'query': 'Structure.get_space_group_info'}}
TOOL RESULT:
 ERROR executing scan_pymatgen_docs: get_pymatgen_docs() missing 1 required positional argument: 'query'

=== STEP 2 ===
RAW MODEL OUTPUT:
 { "tool": "scan_pymatgen_docs", "parameters": { "query": "Structure" } }
PARSED TOOL CALL:
 {'tool': 'scan_pymatgen_docs', 'parameters': {'query': 'Structure'}}
TOOL RESULT:
 ERROR executing scan_pymatgen_docs: get_pymatgen_docs() missing 1 required positional argument: 'query'

=== STEP 3 ===
RAW MODEL OUTPUT:
 { "tool": "scan_pymatgen_docs", "args": { "query": "Structure" } }
PARSED TOOL CALL:
 {'tool': 'scan_pymatgen_docs', 'args': {'query': 'Structure'}}
TOOL RESULT:
 {'success': False, 'stderr': "No pymatgen entry found matching 'Structure'", 'suggestions': ['set_structure', 'get_structure', 'set_